
<div style="text-align: center; padding: 40px 0; border-bottom: 2px solid #e0e0e0; margin-bottom: 20px;">
  <h1 style="font-size: 2.5em; font-weight: 700; color: #000000; margin: 0;">
    🇵🇱 Poland Public Transport
  </h1>
  <h2 style="font-size: 1.4em; font-weight: 300; color: #555; margin: 10px 0 0 0;">
    GTFS Dataset Analysis — 2026
  </h2>
  <p style="color: #999; margin-top: 12px; font-size: 0.9em;">
    GTFS Source: mkuran.pl/gtfs · Provider: Mikołaj Kuranowski (community) · Schedules: PKP PLK · Format: GTFS only — NeTEx not available
  </p>
</div>

## Data Sources

- **GTFS:** `polish_trains.zip` from https://mkuran.pl/gtfs/, maintained by Mikołaj Kuranowski
  - Covers all major Polish rail operators: PolRegio, PKP Intercity, Koleje Mazowieckie, PKP SKM Trójmieście, Koleje Śląskie, Koleje Dolnośląskie, Koleje Wielkopolskie, SKM Warszawa, Łódzka Kolej Aglomeracyjna, Koleje Małopolskie, Arriva RP, RegioJet, Leo Express
  - Schedules sourced from PKP PLK (national rail infrastructure manager) open data API
  - License: PKP PLK public sector data usage terms; Koleje Mazowieckie public sector data usage terms
  - **Note:** This is a community-maintained feed, not an official government publication. Known data caveats are documented in the [PolishTrainsGTFS README](https://github.com/MKuranowski/PolishTrainsGTFS?tab=readme-ov-file#data-caveats).
- **NeTEx:** Not available. Poland's NAP (`przyjazdy.pl`), published by the Ministry of Infrastructure, does not provide a publicly downloadable NeTEx dataset. 

# Part 1: GTFS Exploration

All third-party and standard-library imports used in this notebook are consolidated here.

In [1]:
from pathlib import Path
import zipfile
import pandas as pd

In [2]:
# Set path

data_dir = Path("data")

gtfs_zip_path = data_dir / "polish_trains.zip"

print("GTFS exists:", gtfs_zip_path.exists(), gtfs_zip_path)

GTFS exists: True data/polish_trains.zip


In [3]:
# Inspect GTFS ZIP contents
with zipfile.ZipFile(gtfs_zip_path, "r") as z:
    gtfs_files = pd.DataFrame([
        {
            "name":    info.filename,
            "size_mb": round(info.file_size / (1024 * 1024), 2)
        }
        for info in z.infolist()
    ])

display(gtfs_files.sort_values("size_mb", ascending=False))

,name,size_mb
9,shapes.txt,65.06
6,stop_times.txt,28.93
8,trips.txt,1.84
2,calendar_dates.txt,0.19
5,stops.txt,0.18
7,transfers.txt,0.05
4,routes.txt,0.01
0,agency.txt,0.00
1,attributions.txt,0.00
3,feed_info.txt,0.00


In [4]:
# Helper function to read GTFS tables from the ZIP

def read_gtfs_from_zip(zip_path: Path, filename: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

        # If the exact file name is present, use it directly
        if filename in names:
            target = filename
        else:
            # Otherwise, search for it anywhere inside the ZIP
            matches = [n for n in names if n.lower().endswith("/" + filename.lower())]

            if not matches:
                raise FileNotFoundError(f"{filename} not found in ZIP. Example entries: {names[:20]}")
            if len(matches) > 1:
                raise FileNotFoundError(f"Multiple matches found for {filename}: {matches}")

            target = matches[0]

        with z.open(target) as f:
            return pd.read_csv(f, low_memory=False)

In [5]:
# Load core GTFS tables
stops          = read_gtfs_from_zip(gtfs_zip_path, "stops.txt")
routes         = read_gtfs_from_zip(gtfs_zip_path, "routes.txt")
trips          = read_gtfs_from_zip(gtfs_zip_path, "trips.txt")
calendar_dates = read_gtfs_from_zip(gtfs_zip_path, "calendar_dates.txt")

print("stops:",          stops.shape)
print("routes:",         routes.shape)
print("trips:",          trips.shape)
print("calendar_dates:", calendar_dates.shape)

stops: (3281, 9)
routes: (180, 7)
trips: (22634, 10)
calendar_dates: (10205, 3)


In [6]:
print("stops columns:")
print(list(stops.columns))
display(stops.head())

print("routes columns:")
print(list(routes.columns))
display(routes.head())

print("trips columns:")
print(list(trips.columns))
display(trips.head())

print("calendar_dates columns:")
print(list(calendar_dates.columns))
display(calendar_dates.head())

stops columns:
['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'location_type', 'parent_station', 'stop_timezone', 'country', 'plk_secondary_id']


,stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station,stop_timezone,country,plk_secondary_id
0,0,Lotnisko Modlin,52.445921,20.650717,0,NaN,NaN,NaN,NaN
1,10009,Korsze,54.172188,21.136514,0,NaN,NaN,NaN,NaN
2,10025,Łankiejmy,54.142257,21.069488,0,NaN,NaN,NaN,NaN
3,10033,Sątopy-Samulewo,54.073277,21.024231,0,NaN,NaN,NaN,NaN
4,1008,Świnoujście,53.904708,14.266563,0,NaN,NaN,NaN,NaN


routes columns:
['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'route_color', 'route_text_color']


,route_id,agency_id,route_short_name,route_long_name,route_type,route_color,route_text_color
0,AR,AR,AR,Arriva,2,33CAD6,000000
1,AR_BUS,AR,ZKA AR,Arriva (Zastępcza Komunikacja Autobusowa),3,DE4E4E,FFFFFF
2,IC_EC,IC,EC,EuroCity,2,9D740F,FFFFFF
3,IC_EIC,IC,EIC,Express InterCity,2,898989,000000
4,IC_EIP,IC,EIP,Express InterCity Premium,2,002664,FFFFFF


trips columns:
['trip_id', 'route_id', 'service_id', 'direction_id', 'shape_id', 'trip_short_name', 'trip_headsign', 'plk_category_code', 'plk_train_number', 'plk_train_name']


,trip_id,route_id,service_id,direction_id,shape_id,trip_short_name,trip_headsign,plk_category_code,plk_train_number,plk_train_name
0,KM_900_1,KM_ZL,KM_0,1.0,BUS_0,900,Modlin,NaN,NaN,NaN
1,KM_900_2,KM_ZL,KM_1,1.0,BUS_0,900,Modlin,NaN,NaN,NaN
2,KM_900_3,KM_ZL,KM_2,1.0,BUS_0,900,Modlin,NaN,NaN,NaN
3,KM_901_1,KM_ZL,KM_3,0.0,BUS_1,901,Lotnisko Modlin,NaN,NaN,NaN
4,KM_901_2,KM_ZL,KM_1,0.0,BUS_1,901,Lotnisko Modlin,NaN,NaN,NaN


calendar_dates columns:
['date', 'service_id', 'exception_type']


,date,service_id,exception_type
0,20251214,KM_0,1
1,20251215,KM_0,1
2,20251216,KM_0,1
3,20251217,KM_0,1
4,20251218,KM_0,1


## Comment

The GTFS tables contain the key information needed for a basic exploration.

- `stops.txt` provides stop names and coordinates
- `routes.txt` includes both rail and replacement bus services
- `trips.txt` contains train identifiers and category information
- `calendar_dates.txt` defines the operating dates for services


## Inspect stations

Check:

- how many stops are included
- whether stop names are available
- whether coordinates are available
- whether the feed distinguishes between stations and platforms using `location_type`


In [7]:
station_summary = pd.DataFrame({
    "metric": [
        "Total stop rows",
        "Unique stop IDs",
        "Unique stop names",
        "Rows with missing stop name",
        "Rows with missing coordinates",
        "Unique location_type values"
    ],
    "value": [
        len(stops),
        stops["stop_id"].nunique(),
        stops["stop_name"].nunique(),
        stops["stop_name"].isna().sum(),
        stops[["stop_lat", "stop_lon"]].isna().any(axis=1).sum(),
        stops["location_type"].nunique() if "location_type" in stops.columns else "column missing"
    ]
})

station_summary

,metric,value
0,Total stop rows,3281
1,Unique stop IDs,3281
2,Unique stop names,3155
3,Rows with missing stop name,0
4,Rows with missing coordinates,0
5,Unique location_type values,2


In [8]:
# Check location_type distribution 
# shows how many rows are normal stops, stations, platforms, or missing values

stops["location_type"].value_counts(dropna=False).reset_index()

,location_type,count
0,0,3226
1,1,55


In [9]:
# Inspect stop examples

stops[
    ["stop_id", "stop_name", "stop_lat", "stop_lon", "location_type"]
].head(20)

,stop_id,stop_name,stop_lat,stop_lon,location_type
0,0,Lotnisko Modlin,52.445921,20.650717,0
1,10009,Korsze,54.172188,21.136514,0
2,10025,Łankiejmy,54.142257,21.069488,0
3,10033,Sątopy-Samulewo,54.073277,21.024231,0
4,1008,Świnoujście,53.904708,14.266563,0
5,1024,Międzyzdroje,53.924290,14.455277,0
6,10264,Tołkiny,54.117580,21.235391,0
7,10272,Linkowo,54.090542,21.265414,0
8,10280,Nowy Młyn,54.066465,21.327579,0
9,1032,Lubiewo,53.915513,14.417010,0


## Comment

- The Polish GTFS feed contains **3,281 stop rows**

- All stop IDs are unique, because there are also **3,281 unique stop IDs**

- The feed contains **3,155 unique stop names**, so some stop names appear more than once

- There are **no missing stop names** and **no missing coordinates**

- The `location_type` column has two values:
  - `0`: normal stop/platform where vehicles can stop
  - `1`: parent station, which groups several stops or platforms together

- Most records have `location_type = 0` (**3,226 rows**), while only **55 rows** have `location_type = 1`. This means that most station information is stored as normal stop entries, not as parent station entries.
Since this is a GTFS-only inspection and there is no NeTEx station table to compare against, no filtering is applied.




## Routes

The next part inspects the route information.

The goal is to check:

- how many route rows are included
- whether route names are available
- whether route identifiers are unique
- which `route_type` values appear
- whether the feed contains only rail routes or also other transport types

In [10]:
# Inspect GTFS routes

route_summary = pd.DataFrame({
    "metric": [
        "Total route rows",
        "Unique route IDs",
        "Unique route short names",
        "Unique route long names",
        "Rows with missing route_short_name",
        "Rows with missing route_long_name",
        "Unique route_type values",
        "Unique agency IDs"
    ],
    "value": [
        len(routes),
        routes["route_id"].nunique(),
        routes["route_short_name"].nunique(dropna=True),
        routes["route_long_name"].nunique(dropna=True),
        routes["route_short_name"].isna().sum(),
        routes["route_long_name"].isna().sum(),
        routes["route_type"].nunique(dropna=True),
        routes["agency_id"].nunique(dropna=True)
    ]
})

route_summary

,metric,value
0,Total route rows,180
1,Unique route IDs,180
2,Unique route short names,169
3,Unique route long names,168
4,Rows with missing route_short_name,0
5,Rows with missing route_long_name,0
6,Unique route_type values,2
7,Unique agency IDs,13


In [11]:
# GTFS route_type labels
route_type_labels = {
    "0": "Tram",
    "1": "Metro",
    "2": "Rail",
    "3": "Bus",
    "4": "Ferry",
    "5": "Cable Tram",
    "6": "Aerial Lift",
    "7": "Funicular",
    "11": "Trolleybus",
    "12": "Monorail",
}

route_type_distribution = (
    routes["route_type"]
    .value_counts(dropna=False)
    .reset_index()
)
route_type_distribution.columns = ["route_type", "n_routes"]
route_type_distribution["route_type"] = route_type_distribution["route_type"].astype(str)
route_type_distribution["mode"] = route_type_distribution["route_type"].map(route_type_labels).fillna("Unknown")
route_type_distribution = route_type_distribution[["route_type", "mode", "n_routes"]]
display(route_type_distribution)

,route_type,mode,n_routes
0,2,Rail,141
1,3,Bus,39


In [12]:
# Inspect route examples

routes[
    ["route_id", "agency_id", "route_short_name", "route_long_name", "route_type"]
].head(20)

,route_id,agency_id,route_short_name,route_long_name,route_type
0,AR,AR,AR,Arriva,2
1,AR_BUS,AR,ZKA AR,Arriva (Zastępcza Komunikacja Autobusowa),3
2,IC_EC,IC,EC,EuroCity,2
3,IC_EIC,IC,EIC,Express InterCity,2
4,IC_EIP,IC,EIP,Express InterCity Premium,2
5,IC_IC,IC,IC,InterCity,2
6,IC_ICN,IC,ICN,Retro Nieśpieszny,2
7,IC_IC_BUS,IC,ZKA IC,InterCity (Zastępcza Komunikacja Autobusowa),3
8,IC_MP,IC,MP,Międzynarodowy Pospieszny,2
9,IC_TLK,IC,TLK,Twoje Linie Kolejowe,2


In [13]:
# Inspect repeated route_short_name
routes["route_short_name"].value_counts().loc[lambda x: x > 1]

route_short_name
S1          3
S4          2
S3          2
SKA2        2
ZKA SKA3    2
ZKA K63     2
SKA3        2
K7P         2
PKM1        2
PKM4        2
Name: count, dtype: int64

In [14]:
# Check repeated route_long_name
routes["route_long_name"].value_counts().loc[lambda x: x > 1]

route_long_name
Warszawa — Działdowo                                                                2
Warszawa — Skarżysko-Kamienna                                                       2
Tarnów — Kraków Główny — Trzebinia — Oświęcim (Zastępcza Komunikacja Autobusowa)    2
Warszawa — Małkinia                                                                 2
Sędziszów — Kraków Główny — Skawina — Oświęcim                                      2
Tarnów — Kraków Główny — Trzebinia — Oświęcim                                       2
Kraków Główny — Kraków Nowa Huta — Podłęże (Zastępcza Komunikacja Autobusowa)       2
Kraków Główny — Tarnów — Nowy Sącz — Krynica-Zdrój (Przyspieszony)                  2
Gniezno — Poznań — Kościan                                                          2
Wronki — Poznań — Środa Wlkp.                                                       2
Warszawa — Skarżysko-Kamienna (Przyspieszony)                                       2
Warszawa — Działdowo (Przyspieszony)  

## Comment

- The Polish GTFS feed contains **180 routes** and all of them have unique IDs

- There are no missing route names:
  - **0 missing `route_short_name` values**
  - **0 missing `route_long_name` values**

- The number of unique route names is slightly lower than the number of route rows:
  - **169 unique short route names**
  - **168 unique long route names**

- This means that some public route names appear more than once like for example, the short route name `S1` appears 3 times, while names like `SKA2`, `S4`, `SKA3`, and `S3` appear 2 times.
Some long route names also appear more than once, for example `Warszawa – Działdowo` and `Sędziszów – Kraków Główny – Skawina – Oświęcim`. This is not necessarily a problem, because `route_id` is the real technical identifier. The route name is only a public label and can be reused for more than one technical route row.

- The `route_type` column shows two transport types:
  - `2`: rail
  - `3`: bus

- Most routes are rail routes: **141 routes**

- There are also **39 bus routes**, which include replacement bus services because their names contain `ZKA` or `Zastępcza Komunikacja Autobusowa`. In Polish, `Zastępcza Komunikacja Autobusowa` means replacement bus transport.



## Calendar

The next part inspects the calendar information. 

Since `calendar.txt` is not available, the feed uses `calendar_dates.txt` to describe service operation. Therefore, the goal is to check:

- how many service-date rows are included
- how many unique `service_id` values appear
- what date range is covered
- which `exception_type` values are used
- whether the `service_id` values used in `trips.txt` also appear in `calendar_dates.txt`

In [15]:
# Prepare calendar_dates date column

calendar_dates["date"] = pd.to_datetime(
    calendar_dates["date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

trip_service_ids = set(trips["service_id"].dropna().astype(str))
calendar_service_ids = set(calendar_dates["service_id"].dropna().astype(str))

calendar_summary = pd.DataFrame({
    "metric": [
        "calendar_dates rows",
        "Unique service IDs in trips",
        "Unique service IDs in calendar_dates",
        "Service IDs in trips missing from calendar_dates",
        "Service IDs in calendar_dates not used in trips",
        "Rows with invalid dates",
        "Start date",
        "End date",
        "Unique exception_type values"
    ],
    "value": [
        len(calendar_dates),
        len(trip_service_ids),
        len(calendar_service_ids),
        len(trip_service_ids - calendar_service_ids),
        len(calendar_service_ids - trip_service_ids),
        calendar_dates["date"].isna().sum(),
        calendar_dates["date"].min(),
        calendar_dates["date"].max(),
        calendar_dates["exception_type"].nunique(dropna=True)
    ]
})

calendar_summary

,metric,value
0,calendar_dates rows,10205
1,Unique service IDs in trips,983
2,Unique service IDs in calendar_dates,983
3,Service IDs in trips missing from calendar_dates,0
4,Service IDs in calendar_dates not used in trips,0
5,Rows with invalid dates,0
6,Start date,2025-12-14 00:00:00
7,End date,2026-08-29 00:00:00
8,Unique exception_type values,1


In [16]:
# Check exception_type distribution

exception_type_distribution = (
    calendar_dates["exception_type"]
    .value_counts(dropna=False)
    .reset_index()
)

exception_type_distribution.columns = ["exception_type", "n_rows"]

exception_type_distribution

,exception_type,n_rows
0,1,10205


- `1` means service is added / operates on that date
- `2` means service is removed / does not operate on that date

## Comment

- The Polish GTFS feed contains **10,205 rows** in `calendar_dates.txt`

- There are **983 unique service IDs** in `trips.txt` and also **983 unique service IDs** in `calendar_dates.txt`

- No service IDs are missing from either side. This means every service used by a trip has a matching calendar-date entry

- There are **no invalid dates**

- The calendar covers the period from **2025-12-14** to **2026-08-29**

- Only one `exception_type` value appears:
  - `1`: the service operates on that date

- There are no rows with `exception_type = 2`, so the file does not list removed services

- Since `calendar.txt` is not available, the feed describes service operation directly through specific dates in `calendar_dates.txt`

Overall, the calendar information is consistent.


## Summary table

| Part | What was checked | Main finding | Interpretation |
|---|---|---|---|
| Data availability | Search for usable NeTEx XML data | No usable public NeTEx XML dataset was found | GTFS–NeTEx comparison is not possible |
| Stations | GTFS stop IDs, names, coordinates, and `location_type` | 3,281 stops, no missing names, no missing coordinates | Station data is complete and usable on the GTFS side |
| Routes | GTFS route IDs, names, agencies, and `route_type` | 180 routes, no missing route names, 141 rail routes and 39 bus routes | Route data is complete, the feed includes both rail and bus entries |
| Calendar | `calendar_dates.txt`, service IDs, date range, and `exception_type` | 983 service IDs in trips and calendar_dates, no missing links, date range 2025-12-14 to 2026-08-29 | Calendar data is consistent and usable on the GTFS side |
| Overall result | GTFS-only inspection | The GTFS feed is usable, but NeTEx is missing | Poland is documented as a GTFS-only case, not as a full comparison case |

